# 🏥 Medical Prescription OCR Evaluation
## DeepEval × Ollama — Benchmarking 10 Sub-8B Models

This notebook evaluates the ability of 10 small Ollama models to extract structured entities from noisy medical prescription OCR text and return valid JSON.

**Pipeline:**
```
Raw OCR text → Ollama model → JSON output → DeepEval scoring
```

**Entities extracted:** patient name, date, doctor name, medications (name, dosage, frequency, duration), diagnoses, refills


---
## 0. Installation

In [ ]:
# Install required packages
!pip install deepeval rich pandas tabulate --quiet
!sudo apt-get install zstd && !sudo apt install pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
/bin/bash: line 1: !sudo: command not found
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
#####################################                                     52.7%19.2%

---
## 1. Configuration — Models & Prompt

In [15]:
!ollama serve

time=2026-04-26T10:57:36.769Z level=INFO source=routes.go:1752 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/root/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* file://* tauri://* vscode-webvie

In [2]:
import os 

#─────────────────────────────────────────────────────────────
# PULLING MODELS
#-─────────────────────────────────────────────────────────────

QWEN_MODELS = [
    # Qwen 1
    "qwen:0.5b", 
    "qwen:1.8b",
    "qwen:4b",
    "qwen:7b",

    # Qwen 2
    "qwen2:0.5b", 
    "qwen2:1.5b",
    "qwen2:7b",

    # Qwen 2.5
    "qwen2.5:0.5b",
    "qwen2.5:1.5b",
    "qwen2.5:3b",
    "qwen2.5:7b",
]

# run "ollama serve" in Colab terminal before executing this cell

for m in QWEN_MODELS:
    os.system(f"ollama pull {m}")

!ollama list


NAME            ID              SIZE      MODIFIED               
qwen2.5:7b      845dbda0ea48    4.7 GB    Less than a second ago    
qwen2.5:3b      357c53fb659c    1.9 GB    42 seconds ago            
qwen2.5:1.5b    65ec06548149    986 MB    About a minute ago        
qwen2.5:0.5b    a8b0c5157701    397 MB    About a minute ago        
qwen2:7b        dd314f039b9d    4.4 GB    About a minute ago        
qwen2:1.5b      f6daf2b25194    934 MB    2 minutes ago             
qwen2:0.5b      6f48b936a09f    352 MB    2 minutes ago             
qwen:7b         2091ee8c8d8f    4.5 GB    2 minutes ago             
qwen:4b         d53d04290064    2.3 GB    3 minutes ago             
qwen:1.8b       b6e8ec2e7126    1.1 GB    3 minutes ago             
qwen:0.5b       b5dc5e784f2a    394 MB    3 minutes ago             


In [3]:
import os
# ─────────────────────────────────────────────────────────────
# MODEL LIST  (all < 8B parameters, popular on Ollama)
# ─────────────────────────────────────────────────────────────

MODELS = [
    QWEN_MODELS
]

OLLAMA_BASE_URL = "http://localhost:11434"  # Change if Ollama runs on a different host
OLLAMA_TIMEOUT  = 120   # seconds per inference call

model_number = 0

print("Models to be evaluated:")
for model_list in MODELS:
    for model in model_list:
        model_number += 1
        print(f" - {model}")
print(f"\nTotal models: {model_number}")

Models to be evaluated:
 - qwen:0.5b
 - qwen:1.8b
 - qwen:4b
 - qwen:7b
 - qwen2:0.5b
 - qwen2:1.5b
 - qwen2:7b
 - qwen2.5:0.5b
 - qwen2.5:1.5b
 - qwen2.5:3b
 - qwen2.5:7b

Total models: 11


In [ ]:
# ─────────────────────────────────────────────────────────────
# SYSTEM PROMPT  — instructs the model to extract entities
# ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are a medical data extraction assistant. 
You receive raw OCR text from a medical prescription and must extract structured information.
You work in a context where documents can have both Arabic and French text. Names are Algerian.

Return ONLY a valid JSON object — no markdown, no explanation, no extra text.

Use this exact JSON template:

{
  "medical_facility": {
    "raw_text": string or null,
    "structured": {
      "name": string or null,
      "type": string or null
    }
  },

  "patient": {
    "raw_text": string or null,
    "structured": {
      "full_name": string or null,
      "first_name": string or null,
      "family_name": string or null,

      "date_of_birth": string (YYYY-MM-DD) or null,
      "age": integer or null
    }
  },

  "prescription_date": string (YYYY-MM-DD) or null,
  "doctor": {
    "raw_text": string or null,
    "structured": {
      "name": string or null,
      "specialty": string or null,
      "license_number": string or null
    }
  },

  "medications": [
    {
      "name": string,
      "dosage": string or null,
      "form": string or null,
      "frequency": string or null,
      "duration": string or null,
      "quantity": string or null
    }
  ],
  
}

Rules:
- Always return a valid JSON object that adheres to the template.
- The "raw_text" fields should contain the exact substring from the OCR text that corresponds to the entity, or null if not found.
- The "structured" fields should contain the cleaned and structured information extracted from the raw text.

- It is possible for the medical facility's name to be in Arabic, French, or a combination of both. Extract as much as possible.

- For the patient's name, if possible, split into first name and family name. If not possible, put the full name in "full_name" and leave the others null.
- For the patient's age, if only the date of birth is found, leave the age field null. If only the age is found, leave the date of birth field null. If both are found, fill both fields.

- For the doctor's license number, look for any numeric string that could correspond to the "numéro d'ordre" and return it as is.
- For medications, try to extract as much information as possible for each field. If a field is not mentioned, use null.
- If the OCR text contains multiple medications, return them as separate items in the "medications" list.
- medications must always be a list, even if only one item.
- Normalise dates to YYYY-MM-DD when possible.
- If a field is not found, use null (not empty string).


"""

USER_TEMPLATE = """Extract the structured data from this OCR prescription text:

---
{ocr_text}
---"""

print("Prompt configured.")

---
## 2. Test Dataset — OCR Examples with Ground Truth

In [4]:
#-----------------------------------------------------
# running mass OCR on all the prescriptions pictures
# ----------------------------------------------------

!pip install gdown --quiet

import gdown

# The folder URL or ID from Google Drive
url = "https://drive.google.com/drive/folders/16FthvW2bPYRqt9PyrMI36Mmgruepr_ap?usp=drive_link"

# Downloads the entire folder and its contents
gdown.download_folder(url, quiet=True, use_cookies=False)

!ls # making sure that 'tier_payant_ordonnances'folder is downloaded and present in the current directory 

import pathlib
BASE_PRESCRIPTION_FOLDER = pathlib.Path("tier_payant_ordonnances")

prescriptions_files_paths = list(BASE_PRESCRIPTION_FOLDER.glob("*recto.jpg"))

print("Total number of prescription files:", len(prescriptions_files_paths))
print("---")

for path in prescriptions_files_paths:
    print(path)


sample_data  tier_payant_ordonnances
Total number of prescription files: 7
---
tier_payant_ordonnances/ordo1_recto.jpg
tier_payant_ordonnances/ordo7_recto.jpg
tier_payant_ordonnances/ordo2_recto.jpg
tier_payant_ordonnances/ordo6_recto.jpg
tier_payant_ordonnances/ordo4_recto.jpg
tier_payant_ordonnances/ordo3_recto.jpg
tier_payant_ordonnances/ordo5_recto.jpg


In [12]:
!pip install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
!python -c "import paddle; print(paddle.__version__)"
!pip install paddleocr

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu118/
Error: Can not import paddle core while this file exists: /usr/local/lib/python3.12/dist-packages/paddle/base/libpaddle.so
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/local/lib/python3.12/dist-packages/paddle/__init__.py", line 44, in <module>
    from .base import core  # noqa: F401
    ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/paddle/base/__init__.py", line 38, in <module>
    from . import (  # noqa: F401
  File "/usr/local/lib/python3.12/dist-packages/paddle/base/backward.py", line 28, in <module>
    from . import core, framework, log_helper, unique_name
  File "/usr/local/lib/python3.12/dist-packages/paddle/base/core.py", line 390, in <module>
    raise e
  File "/usr/local/lib/python3.12/dist-packages/paddle/base/core.py", line 267, in <module>
    from . import libpaddle
ImportError: libcuda.so.1: cannot open shared object file: No s

In [16]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(lang='fr')

ordos = {}

for key_raw, path in prescriptions_files_paths.items():
    key = path.name.removesuffix("_recto.jpg")
    ordos[key] = ocr.predict(str(path))  # pass as string to be safe


RuntimeError: PDX has already been initialized. Reinitialization is not supported.

In [ ]:
import json

# ─────────────────────────────────────────────────────────────
# EXAMPLES  — raw OCR text + expected JSON ground truth
# Simulate realistic OCR noise (typos, garbled chars, odd spacing)
# ─────────────────────────────────────────────────────────────

EXAMPLES = [
    # ── Example 1: Clean prescription ──────────────────────────
    {
        "id": "ex_001",
        "description": "Clean prescription, single medication",
        "ocr_text": """
Dr. Sarah Johnson, MD - Cardiology
License: CA-98234
Date: 03/15/2024

Patient: James Carter   DOB: 1978-06-22   ID: PT-44521

Rx:
Lisinopril 10mg tablets
Take 1 tablet by mouth once daily
Qty: 30 tablets  |  Refills: 3

Diagnosis: Hypertension (I10)
""",
        "ground_truth": {
            "patient": {"name": "James Carter", "date_of_birth": "1978-06-22", "id": "PT-44521"},
            "prescription_date": "2024-03-15",
            "doctor": {"name": "Sarah Johnson", "specialty": "Cardiology", "license_number": "CA-98234"},
            "medications": [
                {"name": "Lisinopril", "dosage": "10mg", "form": "tablets",
                 "frequency": "once daily", "duration": None, "quantity": "30 tablets"}
            ],
            "diagnoses": ["Hypertension"],
            "refills": 3,
            "notes": None
        }
    },

    # ── Example 2: Noisy OCR, multiple medications ──────────────
    {
        "id": "ex_002",
        "description": "Noisy OCR, two medications",
        "ocr_text": """
C1inic Med icale du Centre
Dr. Ahmed B0uaza  Specialite: Med.Generale  Lic# MG-00712

Date: 22 jan 2O24  [OCR: O=zero confusion]

Patient: Fatima Zah ra Bensalem   N-SS: FZB-8801
Nee le: 05/11/1990

ORDONNANCE:
1) Amoxici1line 500 mg gelules
   3 fois/jour pendant 7 jours
   Qte: 21 gelules

2) lbuprof ene 400mg comprime
   2 fois par jour si douleur
   Qte: 14 comp  Renouvellement: 0

Diagnostic: Angine bacterienne
Note: Eviter alcool pendant le traitement
""",
        "ground_truth": {
            "patient": {"name": "Fatima Zahra Bensalem", "date_of_birth": "1990-11-05", "id": "FZB-8801"},
            "prescription_date": "2024-01-22",
            "doctor": {"name": "Ahmed Bouaza", "specialty": "Médecine Générale", "license_number": "MG-00712"},
            "medications": [
                {"name": "Amoxicilline", "dosage": "500mg", "form": "gélules",
                 "frequency": "3 fois/jour", "duration": "7 jours", "quantity": "21"},
                {"name": "Ibuprofène", "dosage": "400mg", "form": "comprimé",
                 "frequency": "2 fois/jour", "duration": None, "quantity": "14"}
            ],
            "diagnoses": ["Angine bactérienne"],
            "refills": 0,
            "notes": "Eviter alcool pendant le traitement"
        }
    },

    # ── Example 3: Heavily garbled OCR ─────────────────────────
    {
        "id": "ex_003",
        "description": "Heavily garbled OCR, missing fields",
        "ocr_text": """
D r . M i k e  T h o m p s o n   Pedi4trics
D@te: Sept 5 2023

Pat: Emily R0se Nguyen  DOB 2O15-O3-12

Rx1: Amox!cil!in  250 m9  susp3nsion
give 5ml tw!ce da!ly f0r 10 days

Rx2: C3tiriz!ne 5mg syr0p
1 teasp0on at bedtime
30 days supply

Dx: Ot!tis media, A11ergic rhin!tis
Ref!ls: 1
""",
        "ground_truth": {
            "patient": {"name": "Emily Rose Nguyen", "date_of_birth": "2015-03-12", "id": None},
            "prescription_date": "2023-09-05",
            "doctor": {"name": "Mike Thompson", "specialty": "Pediatrics", "license_number": None},
            "medications": [
                {"name": "Amoxicillin", "dosage": "250mg", "form": "suspension",
                 "frequency": "twice daily", "duration": "10 days", "quantity": None},
                {"name": "Cetirizine", "dosage": "5mg", "form": "syrup",
                 "frequency": "at bedtime", "duration": "30 days", "quantity": None}
            ],
            "diagnoses": ["Otitis media", "Allergic rhinitis"],
            "refills": 1,
            "notes": None
        }
    },

    # ── Example 4: Minimal / sparse prescription ───────────────
    {
        "id": "ex_004",
        "description": "Minimal info, one medication, no refills",
        "ocr_text": """
ORDONNANCE MEDICALE
Date: 10-04-2024
Médecin: Dr. Nadia Cherif

Pour: M. Karim Hadj Ali

- Metformine 850mg cp
  1 cp matin et soir pendant 3 mois

Diabète type 2
""",
        "ground_truth": {
            "patient": {"name": "Karim Hadj Ali", "date_of_birth": None, "id": None},
            "prescription_date": "2024-04-10",
            "doctor": {"name": "Nadia Cherif", "specialty": None, "license_number": None},
            "medications": [
                {"name": "Metformine", "dosage": "850mg", "form": "comprimé",
                 "frequency": "matin et soir", "duration": "3 mois", "quantity": None}
            ],
            "diagnoses": ["Diabète type 2"],
            "refills": None,
            "notes": None
        }
    },

    # ── Example 5: Three medications, complex OCR ───────────────
    {
        "id": "ex_005",
        "description": "Three medications, mixed noise",
        "ocr_text": """
Westside Medical Group
Dr. Priya Sharma  |  Internal Medicine  |  Lic: TX-55309
Date: January 18, 2024    Patient ID: WMG-2291

Patient: Robert Klein   Born: 1965-09-30

1. Atorvastatin 40 mg tablet
   Once at bedtime  x  90 days  Qty: 90  Refill: 2

2. Metop ro lol Succinate 50mg XL tab
   1 tab QD  x  90 days  Qty: 90

3. Aspirin 81 mg EC tab
   1 tab daily with food  Qty: 90

Dx: Hyperlipidemia, HTN, CAD (preventive)
Note: F/U in 3 months with lipid panel
""",
        "ground_truth": {
            "patient": {"name": "Robert Klein", "date_of_birth": "1965-09-30", "id": "WMG-2291"},
            "prescription_date": "2024-01-18",
            "doctor": {"name": "Priya Sharma", "specialty": "Internal Medicine", "license_number": "TX-55309"},
            "medications": [
                {"name": "Atorvastatin", "dosage": "40mg", "form": "tablet",
                 "frequency": "once at bedtime", "duration": "90 days", "quantity": "90"},
                {"name": "Metoprolol Succinate", "dosage": "50mg", "form": "tablet",
                 "frequency": "once daily", "duration": "90 days", "quantity": "90"},
                {"name": "Aspirin", "dosage": "81mg", "form": "tablet",
                 "frequency": "daily with food", "duration": None, "quantity": "90"}
            ],
            "diagnoses": ["Hyperlipidemia", "Hypertension", "CAD"],
            "refills": 2,
            "notes": "F/U in 3 months with lipid panel"
        }
    },
]

print(f"Loaded {len(EXAMPLES)} test examples.")
for ex in EXAMPLES:
    print(f"  [{ex['id']}] {ex['description']}")

---
## 3. Ollama Inference Helper

In [ ]:
import ollama
import json
import re
import time

def call_ollama(model: str, ocr_text: str) -> dict:
    """
    Send OCR text to an Ollama model and return:
      - raw_output (str)
      - parsed_json (dict | None)
      - latency_sec (float)
      - error (str | None)
    """
    user_msg = USER_TEMPLATE.format(ocr_text=ocr_text)
    t0 = time.time()
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
            options={"temperature": 0.0},  # Deterministic for eval
        )
        latency = time.time() - t0
        raw = response["message"]["content"].strip()

        # Strip markdown fences if model wrapped output
        clean = re.sub(r"^```[a-z]*\n?", "", raw)
        clean = re.sub(r"\n?```$", "", clean).strip()

        parsed = json.loads(clean)
        return {"raw_output": raw, "parsed_json": parsed, "latency_sec": latency, "error": None}

    except json.JSONDecodeError as e:
        return {"raw_output": raw if 'raw' in dir() else "", "parsed_json": None,
                "latency_sec": time.time() - t0, "error": f"JSON parse error: {e}"}
    except Exception as e:
        return {"raw_output": "", "parsed_json": None,
                "latency_sec": time.time() - t0, "error": str(e)}


def check_models_available(models: list) -> dict:
    """Check which models are actually pulled in Ollama."""
    try:
        available = {m["name"] for m in ollama.list()["models"]}
    except Exception as e:
        print(f"⚠️  Could not connect to Ollama: {e}")
        return {m: False for m in models}
    status = {}
    for m in models:
        found = m in available or any(m in a for a in available)
        status[m] = found
        icon = "✅" if found else "❌"
        print(f"  {icon} {m}")
    return status

print("Checking model availability...")
model_status = check_models_available(MODELS)
AVAILABLE_MODELS = [m for m, ok in model_status.items() if ok]
print(f"\n{len(AVAILABLE_MODELS)}/{len(MODELS)} models ready.")

---
## 4. Custom DeepEval Metrics

In [ ]:
from deepeval.metrics import BaseMetric
from deepeval.test_case import LLMTestCase
import math

# ─────────────────────────────────────────────────────────────
# Helper: fuzzy string match (normalise before comparing)
# ─────────────────────────────────────────────────────────────
def normalise(s) -> str:
    if s is None:
        return ""
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def str_match(a, b) -> bool:
    return normalise(a) == normalise(b)

def partial_match(pred, gold) -> float:
    """Score 0–1 for fuzzy string similarity."""
    p, g = normalise(pred), normalise(gold)
    if not g:
        return 1.0 if not p else 0.5   # null expected: null predicted = perfect
    if not p:
        return 0.0
    if p == g:
        return 1.0
    # Longest common subsequence ratio
    from difflib import SequenceMatcher
    return SequenceMatcher(None, p, g).ratio()


# ─────────────────────────────────────────────────────────────
# METRIC 1: JSON Validity
# ─────────────────────────────────────────────────────────────
class JSONValidityMetric(BaseMetric):
    """Did the model return parseable JSON at all?"""
    name = "JSON Validity"
    threshold = 1.0

    def measure(self, test_case: LLMTestCase) -> float:
        try:
            parsed = json.loads(test_case.actual_output)
            self.score = 1.0
            self.reason = "Valid JSON"
        except Exception as e:
            self.score = 0.0
            self.reason = f"Invalid JSON: {e}"
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)


# ─────────────────────────────────────────────────────────────
# METRIC 2: Schema Compliance
# ─────────────────────────────────────────────────────────────
REQUIRED_KEYS = {"patient", "prescription_date", "doctor", "medications", "diagnoses", "refills", "notes"}
PATIENT_KEYS  = {"name", "date_of_birth", "id"}
DOCTOR_KEYS   = {"name", "specialty", "license_number"}
MED_KEYS      = {"name", "dosage", "form", "frequency", "duration", "quantity"}

class SchemaComplianceMetric(BaseMetric):
    """Does the JSON match the required schema structure?"""
    name = "Schema Compliance"
    threshold = 0.8

    def measure(self, test_case: LLMTestCase) -> float:
        try:
            data = json.loads(test_case.actual_output)
        except Exception:
            self.score = 0.0; self.reason = "Unparseable JSON"; return 0.0

        checks = []
        checks.append(isinstance(data, dict))
        checks.append(REQUIRED_KEYS.issubset(data.keys()))
        checks.append(isinstance(data.get("patient"), dict) and PATIENT_KEYS.issubset(data["patient"].keys()))
        checks.append(isinstance(data.get("doctor"), dict) and DOCTOR_KEYS.issubset(data["doctor"].keys()))
        checks.append(isinstance(data.get("medications"), list))
        checks.append(isinstance(data.get("diagnoses"), list))
        # Each medication has required keys
        meds = data.get("medications", [])
        if meds:
            checks.append(all(MED_KEYS.issubset(m.keys()) for m in meds))

        self.score = sum(checks) / len(checks)
        self.reason = f"{sum(checks)}/{len(checks)} schema checks passed"
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)


# ─────────────────────────────────────────────────────────────
# METRIC 3: Entity Accuracy  (the core metric)
# ─────────────────────────────────────────────────────────────
class EntityAccuracyMetric(BaseMetric):
    """
    Compares extracted entities against ground truth.
    expected_output must be the ground truth JSON string.
    """
    name = "Entity Accuracy"
    threshold = 0.7

    def measure(self, test_case: LLMTestCase) -> float:
        try:
            pred = json.loads(test_case.actual_output)
        except Exception:
            self.score = 0.0; self.reason = "Unparseable JSON"; return 0.0

        gold = json.loads(test_case.expected_output)
        scores = []

        # Patient fields
        for f in ["name", "date_of_birth", "id"]:
            scores.append(partial_match(
                pred.get("patient", {}).get(f),
                gold.get("patient", {}).get(f)
            ))

        # Prescription date
        scores.append(partial_match(pred.get("prescription_date"), gold.get("prescription_date")))

        # Doctor fields
        for f in ["name", "specialty", "license_number"]:
            scores.append(partial_match(
                pred.get("doctor", {}).get(f),
                gold.get("doctor", {}).get(f)
            ))

        # Medication matching (align by index; penalise missing/extra)
        pred_meds = pred.get("medications", [])
        gold_meds = gold.get("medications", [])
        n = max(len(pred_meds), len(gold_meds))
        med_scores = []
        for i in range(n):
            pm = pred_meds[i] if i < len(pred_meds) else {}
            gm = gold_meds[i] if i < len(gold_meds) else {}
            for f in ["name", "dosage", "frequency", "duration"]:
                med_scores.append(partial_match(pm.get(f), gm.get(f)))
        if med_scores:
            scores.append(sum(med_scores) / len(med_scores))

        # Diagnoses (set overlap)
        pred_dx = {normalise(d) for d in pred.get("diagnoses", [])}
        gold_dx = {normalise(d) for d in gold.get("diagnoses", [])}
        if gold_dx:
            dx_score = len(pred_dx & gold_dx) / len(gold_dx)
        else:
            dx_score = 1.0 if not pred_dx else 0.5
        scores.append(dx_score)

        # Refills
        pred_r = pred.get("refills")
        gold_r = gold.get("refills")
        if gold_r is None:
            scores.append(1.0 if pred_r is None else 0.5)
        else:
            scores.append(1.0 if pred_r == gold_r else 0.0)

        self.score = round(sum(scores) / len(scores), 4)
        self.reason = f"Mean entity match = {self.score:.2%} over {len(scores)} fields"
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)


# ─────────────────────────────────────────────────────────────
# METRIC 4: Medication Name Recall
# ─────────────────────────────────────────────────────────────
class MedicationRecallMetric(BaseMetric):
    """How many medication names from the ground truth did the model find?"""
    name = "Medication Recall"
    threshold = 0.8

    def measure(self, test_case: LLMTestCase) -> float:
        try:
            pred = json.loads(test_case.actual_output)
            gold = json.loads(test_case.expected_output)
        except Exception:
            self.score = 0.0; self.reason = "Parse error"; return 0.0

        gold_names = [normalise(m["name"]) for m in gold.get("medications", [])]
        pred_names = [normalise(m.get("name", "")) for m in pred.get("medications", [])]

        if not gold_names:
            self.score = 1.0; self.reason = "No medications in ground truth"; return 1.0

        found = sum(
            any(partial_match(pn, gn) > 0.8 for pn in pred_names)
            for gn in gold_names
        )
        self.score = round(found / len(gold_names), 4)
        self.reason = f"Found {found}/{len(gold_names)} medications"
        return self.score

    def is_successful(self) -> bool:
        return self.score >= self.threshold

    async def a_measure(self, test_case: LLMTestCase) -> float:
        return self.measure(test_case)


print("✅ All 4 custom metrics defined:")
print("   1. JSONValidityMetric      — did model return parseable JSON?")
print("   2. SchemaComplianceMetric  — does JSON match required schema?")
print("   3. EntityAccuracyMetric    — how accurate are extracted fields?")
print("   4. MedicationRecallMetric  — how many drug names were found?")

---
## 5. Run Evaluation

In [ ]:
from deepeval.test_case import LLMTestCase
from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn, TimeElapsedColumn
from rich.console import Console

console = Console()

METRICS = [
    JSONValidityMetric(),
    SchemaComplianceMetric(),
    EntityAccuracyMetric(),
    MedicationRecallMetric(),
]

# results[model][example_id] = {metric_name: score, latency, error, ...}
results = {}

total_calls = len(AVAILABLE_MODELS) * len(EXAMPLES)
console.print(f"\n[bold cyan]Starting evaluation: {len(AVAILABLE_MODELS)} models × {len(EXAMPLES)} examples = {total_calls} inference calls[/]\n")

with Progress(
    SpinnerColumn(),
    TextColumn("[bold blue]{task.description}"),
    BarColumn(),
    TextColumn("{task.completed}/{task.total}"),
    TimeElapsedColumn(),
    console=console
) as progress:
    task = progress.add_task("Evaluating...", total=total_calls)

    for model in AVAILABLE_MODELS:
        results[model] = {}
        for ex in EXAMPLES:
            progress.update(task, description=f"{model} | {ex['id']}")

            # 1. Inference
            infer = call_ollama(model, ex["ocr_text"])

            # 2. Build test case
            actual = infer["raw_output"] if infer["parsed_json"] is None else json.dumps(infer["parsed_json"])
            test_case = LLMTestCase(
                input=USER_TEMPLATE.format(ocr_text=ex["ocr_text"]),
                actual_output=actual,
                expected_output=json.dumps(ex["ground_truth"]),
            )

            # 3. Score each metric
            metric_scores = {}
            for metric in METRICS:
                metric.measure(test_case)
                metric_scores[metric.name] = round(metric.score, 4)

            results[model][ex["id"]] = {
                "scores": metric_scores,
                "latency": round(infer["latency_sec"], 2),
                "error": infer["error"],
                "parsed": infer["parsed_json"],
                "raw": infer["raw_output"][:300],  # Truncate for storage
            }
            progress.advance(task)

console.print("\n[bold green]✅ Evaluation complete![/]")

---
## 6. Results — Per-Model Summary Table

In [ ]:
import pandas as pd

METRIC_NAMES = [m.name for m in METRICS]

rows = []
for model in AVAILABLE_MODELS:
    model_results = results[model]
    n = len(model_results)
    avg_scores = {mn: 0.0 for mn in METRIC_NAMES}
    avg_latency = 0.0
    errors = 0

    for ex_id, res in model_results.items():
        for mn in METRIC_NAMES:
            avg_scores[mn] += res["scores"].get(mn, 0.0)
        avg_latency += res["latency"]
        if res["error"]:
            errors += 1

    row = {"Model": model}
    for mn in METRIC_NAMES:
        row[mn] = round(avg_scores[mn] / n, 3)
    row["Avg Latency (s)"] = round(avg_latency / n, 2)
    row["Errors"] = errors
    # Overall score = mean of all metrics (excluding latency/errors)
    row["Overall"] = round(sum(row[mn] for mn in METRIC_NAMES) / len(METRIC_NAMES), 3)
    rows.append(row)

df = pd.DataFrame(rows).sort_values("Overall", ascending=False).reset_index(drop=True)
df.index += 1  # 1-based ranking
df.index.name = "Rank"

# Colour-coded display
def highlight(val, threshold=0.7):
    if isinstance(val, float):
        if val >= 0.9: color = "background-color: #2d6a4f; color: white"
        elif val >= 0.7: color = "background-color: #52b788"
        elif val >= 0.5: color = "background-color: #f4a261"
        else: color = "background-color: #e63946; color: white"
        return color
    return ""

styled = df.style.applymap(highlight, subset=METRIC_NAMES + ["Overall"])
styled

---
## 7. Per-Example Breakdown

In [ ]:
# Pivot: for a chosen metric, show all models × all examples
TARGET_METRIC = "Entity Accuracy"  # ← Change to any metric name

pivot_rows = []
for model in AVAILABLE_MODELS:
    row = {"Model": model}
    for ex in EXAMPLES:
        score = results[model][ex["id"]]["scores"].get(TARGET_METRIC, None)
        row[ex["id"]] = round(score, 3) if score is not None else "ERR"
    pivot_rows.append(row)

pivot_df = pd.DataFrame(pivot_rows).set_index("Model")
print(f"\n📊 {TARGET_METRIC} — per-example scores:")
pivot_df.style.applymap(highlight, subset=list(pivot_df.columns))

---
## 8. Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Plot 1: Overall score bar chart ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = plt.cm.RdYlGn(df["Overall"].values)
bars = axes[0].barh(df["Model"], df["Overall"], color=colors, edgecolor="white", height=0.6)
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("Score", fontsize=11)
axes[0].set_title("Overall Score (avg of all metrics)", fontsize=13, fontweight="bold")
axes[0].axvline(0.7, color="orange", linestyle="--", linewidth=1, label="threshold 0.7")
axes[0].legend()
for bar, val in zip(bars, df["Overall"]):
    axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f"{val:.2f}", va="center", fontsize=9)

# ── Plot 2: Latency vs Entity Accuracy scatter ─────────────────
axes[1].scatter(df["Avg Latency (s)"], df["Entity Accuracy"],
                c=df["Overall"], cmap="RdYlGn", s=120, edgecolors="black", linewidths=0.8)
for _, row_ in df.iterrows():
    axes[1].annotate(row_["Model"].split(":")[0],
                     (row_["Avg Latency (s)"], row_["Entity Accuracy"]),
                     textcoords="offset points", xytext=(5, 5), fontsize=8)
axes[1].set_xlabel("Avg Latency (s)", fontsize=11)
axes[1].set_ylabel("Entity Accuracy", fontsize=11)
axes[1].set_title("Quality vs Speed Trade-off", fontsize=13, fontweight="bold")
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("/tmp/eval_plots1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Plot 3: Radar / spider chart per model ─────────────────────
from matplotlib.patches import FancyArrowPatch

RADAR_METRICS = METRIC_NAMES  # all 4 metrics
N = len(RADAR_METRICS)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
cmap = plt.cm.tab10

for i, model in enumerate(AVAILABLE_MODELS):
    vals = [df[df["Model"] == model][mn].values[0] for mn in RADAR_METRICS]
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=1.8, color=cmap(i), label=model)
    ax.fill(angles, vals, alpha=0.08, color=cmap(i))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(RADAR_METRICS, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_title("Model Capability Radar", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)

plt.tight_layout()
plt.savefig("/tmp/eval_plots2.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 9. Inspect Individual Outputs

In [ ]:
from rich.table import Table
from rich.panel import Panel
from rich.syntax import Syntax

# ── Change these to inspect any model/example combination ──────
INSPECT_MODEL   = AVAILABLE_MODELS[0]   # Best model (sorted by Overall)
INSPECT_EXAMPLE = "ex_002"               # The noisy OCR example
# ──────────────────────────────────────────────────────────────

r = results[INSPECT_MODEL][INSPECT_EXAMPLE]
ex = next(e for e in EXAMPLES if e["id"] == INSPECT_EXAMPLE)

console.print(Panel(f"[bold]{INSPECT_MODEL}[/] on [bold]{INSPECT_EXAMPLE}[/] — {ex['description']}",
                    style="cyan"))

# Scores table
t = Table(show_header=True, header_style="bold magenta")
t.add_column("Metric"); t.add_column("Score", justify="right")
for mn, sc in r["scores"].items():
    color = "green" if sc >= 0.7 else "red"
    t.add_row(mn, f"[{color}]{sc:.3f}[/]")
t.add_row("Latency", f"{r['latency']}s")
if r["error"]:
    t.add_row("Error", f"[red]{r['error']}[/]")
console.print(t)

# Side-by-side JSON diff
console.print("\n[bold]Model output:[/]")
if r["parsed"]:
    console.print(Syntax(json.dumps(r["parsed"], indent=2, ensure_ascii=False), "json", theme="monokai"))
else:
    console.print(f"[red]Raw output (not valid JSON):[/]\n{r['raw']}")

console.print("\n[bold]Ground truth:[/]")
console.print(Syntax(json.dumps(ex["ground_truth"], indent=2, ensure_ascii=False), "json", theme="monokai"))

---
## 10. Export Results to CSV

In [ ]:
# ── Summary CSV ────────────────────────────────────────────────
df.to_csv("ocr_eval_summary.csv")
print("Saved: ocr_eval_summary.csv")

# ── Detailed CSV (every model × example × metric) ─────────────
detail_rows = []
for model in AVAILABLE_MODELS:
    for ex in EXAMPLES:
        r = results[model][ex["id"]]
        row = {"model": model, "example_id": ex["id"], "description": ex["description"],
               "latency_s": r["latency"], "error": r["error"]}
        row.update(r["scores"])
        detail_rows.append(row)

detail_df = pd.DataFrame(detail_rows)
detail_df.to_csv("ocr_eval_detailed.csv", index=False)
print("Saved: ocr_eval_detailed.csv")

# ── Print final ranking ────────────────────────────────────────
console.print("\n[bold cyan]🏆 Final Ranking[/]")
for rank, (_, row_) in enumerate(df.iterrows(), 1):
    medal = ["🥇", "🥈", "🥉"].get(rank - 1, "  ")
    console.print(f"{medal} #{rank:2d}  [bold]{row_['Model']:<30}[/]  Overall: {row_['Overall']:.3f}  "
                  f"Entity Accuracy: {row_['Entity Accuracy']:.3f}  "
                  f"Latency: {row_['Avg Latency (s)']}s")

---
## 11. Add Your Own Examples

To extend the benchmark, append to `EXAMPLES` in Cell 3 following this template:

```python
{
    "id": "ex_006",
    "description": "Your description",
    "ocr_text": """< paste raw OCR here >""",
    "ground_truth": {
        "patient": {"name": "...", "date_of_birth": "YYYY-MM-DD", "id": None},
        "prescription_date": "YYYY-MM-DD",
        "doctor": {"name": "...", "specialty": None, "license_number": None},
        "medications": [
            {"name": "...", "dosage": "...", "form": "...",
             "frequency": "...", "duration": None, "quantity": None}
        ],
        "diagnoses": ["..."],
        "refills": None,
        "notes": None
    }
}
```

To add your own models, append to `MODELS` and run `ollama pull <model_name>` first.

---
## 12. Tips & Troubleshooting

| Issue | Fix |
|---|---|
| Model not found | Run `ollama pull <model>` in terminal |
| Timeout errors | Increase `OLLAMA_TIMEOUT` or use a smaller model |
| All JSON Validity = 0 | The model may need `format: "json"` — add `"format": "json"` to `ollama.chat()` options |
| Ollama not running | Start with `ollama serve` in terminal |
| Want LLM-based eval | Replace custom metrics with DeepEval's `AnswerRelevancyMetric` + set `DEEPEVAL_TELEMETRY_OPT_OUT=YES` |
